# RAG Codegen AOSP — LLM-based (Full Run v6)
**Purpose:** Run the complete 4-condition experiment: C1 → C2 → DSPy → AOSP indexing → C3 (RAG+DSPy) → C4 (Feedback).

**Runtime:** Colab A100 High-RAM  
**Model:** Qwen 2.5-coder:32b via Ollama

---
### Pipeline overview
| Phase | What | Script | Status |
|-------|------|--------|--------|
| Setup | Drive, system deps, Ollama, repo | — | Run |
| C1 Baseline | Vanilla LLM generation | `multi_main.py` | **Run live** |
| C2 Adaptive | Adaptive generation | `multi_main_adaptive.py` | **Run live** |
| DSPy Optimiser | MIPRO-auto light, train-size 2 | `dspy_opt/optimizer.py` | **Run live** |
| AOSP Indexing | Clone AOSP → build ChromaDB | `rag.aosp_indexer` | **Run or Restore** |
| Bug fixes | Patch scoring + output cleaning | — | Auto |
| C3 RAG+DSPy | RAG + optimised prompts | `multi_main_rag_dspy.py` | **Run live** |
| C4 Feedback | Generate → validate → refine | `multi_main_c4_feedback.py` | **Run live** |
| Report | diagnose → rescore → compare → analyse | — | Run |

### Known fixes applied (Section 10b)
1. `lstrip("**/")` → `removeprefix("**/")` — glob pattern bug caused HAL scores = 0.000
2. `_clean_output()` added to backend sub-modules — markdown fences caused SyntaxError
3. `output_root=OUTPUT_DIR` passed to architect agent — HAL files landed in wrong directory

## 1. Configuration

In [ ]:
# ── Configuration — edit these paths once ─────────────────────────
import os, time, shutil, glob, subprocess, requests
from pathlib import Path

DRIVE_SRC    = "/content/drive/MyDrive/mse25_thesis"
REPO_URL     = "https://github.com/appdev1307/code-codegen-aosp-llm-based.git"
REPO_DIR     = "/content/code-codegen-aosp-llm-based"
MODEL_NAME   = "qwen2.5-coder:32b"
OLLAMA_LOG   = "/content/ollama.log"

# AOSP source & ChromaDB paths
AOSP_SRC_DIR = f"{REPO_DIR}/aosp_source"
CHROMA_DB    = f"{REPO_DIR}/rag/chroma_db"
CHROMA_ZIP   = f"{DRIVE_SRC}/chroma_db.zip"

# Restore mappings: (drive_zip, local_target_dir)
RESTORE_MAP = {
    "output_c1.zip":          f"{REPO_DIR}/output",
    "output_c2.zip":          f"{REPO_DIR}/output_adaptive",
    "dspy_saved_backup.zip":  f"{REPO_DIR}/dspy_opt/saved",
}

print("✓ Config loaded")

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
Path(DRIVE_SRC).mkdir(parents=True, exist_ok=True)
print(f"✓ Drive mounted — {DRIVE_SRC}")

## 3. System dependencies

In [ ]:
%%capture
!apt-get update -qq
!apt-get install -y -qq clang checkpolicy zstd

for tool in ["clang", "checkpolicy", "zstd"]:
    path = subprocess.run(["which", tool], capture_output=True, text=True).stdout.strip()
    print(f"  {tool}: {'✓ ' + path if path else '❌ NOT FOUND'}")

## 4. Ollama setup

In [ ]:
def start_ollama():
    """Install (if needed), start Ollama, and wait until healthy."""
    if not Path("/usr/local/bin/ollama").exists():
        print("Installing Ollama...")
        subprocess.run("curl -fsSL https://ollama.com/install.sh | sh",
                       shell=True, capture_output=True)
    subprocess.Popen(["ollama", "serve"],
                     stdout=open(OLLAMA_LOG, "w"), stderr=subprocess.STDOUT)
    for i in range(30):
        try:
            r = requests.get("http://localhost:11434/api/tags", timeout=2)
            if r.status_code == 200:
                print(f"✓ Ollama server ready (took {i+1}s)")
                return True
        except Exception:
            pass
        time.sleep(1)
    raise RuntimeError("❌ Ollama failed to start — check ollama.log")

start_ollama()

In [ ]:
# Pull model (skip if already cached)
result = subprocess.run(["ollama", "list"], capture_output=True, text=True)
if MODEL_NAME.split(":")[0] in result.stdout:
    print(f"✓ Model {MODEL_NAME} already available")
else:
    print(f"Pulling {MODEL_NAME}...")
    !ollama pull {MODEL_NAME}
!ollama list

## 5. Clone & setup repository

In [ ]:
if Path(REPO_DIR).exists():
    print("Repo exists — pulling latest...")
    subprocess.run(["git", "stash"], cwd=REPO_DIR, capture_output=True)
    subprocess.run(["git", "pull", "origin", "main"], cwd=REPO_DIR)
else:
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
print(f"✓ Working directory: {os.getcwd()}")

In [ ]:
%%capture
!pip install -q -r requirements.txt
!pip install -q requests pyyaml chromadb sentence-transformers dspy-ai
!pip install -q jinja2 fastapi uvicorn pydantic
print("✓ Python dependencies installed")

## 6. Run C1 — Baseline
> Vanilla LLM code generation. Produces baseline labelled signals for DSPy training.

In [ ]:
start_ollama()
!python multi_main.py

In [ ]:
# Save C1 output to Drive
shutil.make_archive(f"{DRIVE_SRC}/output_c1", "zip", f"{REPO_DIR}/output")
print(f"✓ C1 saved to Drive ({len(list(Path(f'{REPO_DIR}/output').rglob('*')))} files)")

## 7. Run C2 — Adaptive
> Adaptive generation with dynamic prompting. Produces `VSS_LABELLED_50.json` for DSPy.

In [ ]:
start_ollama()
!python multi_main_adaptive.py

In [ ]:
# Save C2 output to Drive
shutil.make_archive(f"{DRIVE_SRC}/output_c2", "zip", f"{REPO_DIR}/output_adaptive")
print(f"✓ C2 saved to Drive ({len(list(Path(f'{REPO_DIR}/output_adaptive').rglob('*')))} files)")

## 8. Run DSPy optimiser (MIPROv2)
> Bayesian search (TPE) over prompt instructions + few-shot demos.
> `train-size 2` for fastest iteration. Uncomment 4/8 for deeper search.

In [ ]:
start_ollama()

#!python dspy_opt/optimizer.py --mipro-auto light --train-size 8 --force
#!python dspy_opt/optimizer.py --mipro-auto light --train-size 4 --force
!python dspy_opt/optimizer.py --mipro-auto light --train-size 2 --force

dspy_count = len(glob.glob("dspy_opt/saved/*/program.json"))
print(f"\n✓ DSPy programs: {dspy_count}")

In [ ]:
# Save DSPy programs to Drive
shutil.make_archive(f"{DRIVE_SRC}/dspy_saved_backup", "zip", f"{REPO_DIR}/dspy_opt/saved")
print("✓ DSPy programs saved to Drive")

## 9. Build ChromaDB from AOSP source (or restore from cache)
> Strategy: restore `chroma_db.zip` from Drive if available, otherwise clone + index.

In [ ]:
# ── 9a. Try restoring ChromaDB from Drive ────────────────────────
chroma_restored = False
if Path(CHROMA_ZIP).exists():
    print(f"Found cached ChromaDB: {CHROMA_ZIP}")
    target = Path(CHROMA_DB)
    if target.exists():
        shutil.rmtree(target)
    target.mkdir(parents=True, exist_ok=True)
    shutil.unpack_archive(CHROMA_ZIP, CHROMA_DB)
    chroma_restored = True
    print("✓ ChromaDB restored from Drive cache")
else:
    print("⚠ No cached ChromaDB — will build from AOSP source")

In [ ]:
# ── 9b. Clone AOSP + build ChromaDB (only if not restored) ───────
if not chroma_restored:
    print("Cloning AOSP source (shallow, ~300 MB)...\n")
    aosp_repos = [
        ("https://android.googlesource.com/platform/hardware/interfaces", "hardware"),
        ("https://android.googlesource.com/platform/system/sepolicy",     "sepolicy"),
        ("https://android.googlesource.com/platform/packages/services/Car", "car"),
    ]
    Path(AOSP_SRC_DIR).mkdir(parents=True, exist_ok=True)
    for url, name in aosp_repos:
        dest = f"{AOSP_SRC_DIR}/{name}"
        if Path(dest).exists():
            print(f"  ✓ {name} already cloned")
            continue
        print(f"  Cloning {name}...")
        r = subprocess.run(["git", "clone", "--depth=1", url, dest], capture_output=True, text=True)
        print(f"  {'✓' if r.returncode == 0 else '❌'} {name}")
    print(f"\n✓ AOSP source ready")
else:
    print("Skipping AOSP clone — ChromaDB restored from cache")

In [ ]:
# ── 9c. Run AOSP indexer → ChromaDB ──────────────────────────────
if not chroma_restored:
    print("Running AOSP indexer...\n")
    !python -m rag.aosp_indexer --source {AOSP_SRC_DIR} --db {CHROMA_DB}
    print("\n✓ AOSP indexing complete")
else:
    print("Skipping indexer — ChromaDB already populated")

In [ ]:
# ── 9d. Verify ChromaDB ──────────────────────────────────────────
import chromadb
client = chromadb.PersistentClient(path=CHROMA_DB)
collections = client.list_collections()
total_chunks = sum(col.count() for col in collections)
print(f"  Collections: {len(collections)}, Total chunks: {total_chunks}")
for col in collections:
    print(f"    → {col.name}: {col.count()}")
assert total_chunks > 0, "❌ ChromaDB is EMPTY — RAG will not work!"
print(f"\n✓ ChromaDB verified")
del client

In [ ]:
# ── 9e. Save ChromaDB to Drive ───────────────────────────────────
if not chroma_restored and not Path(CHROMA_ZIP).exists():
    shutil.make_archive(CHROMA_ZIP.replace(".zip", ""), "zip", CHROMA_DB)
    print(f"✓ Saved to {CHROMA_ZIP}")
else:
    print("ChromaDB cache already exists — skipping save")

## 10. Restore cached outputs & preflight check
> Restores C1/C2/DSPy from Drive if not on disk (e.g. after Colab restart).
> Then verifies all dependencies before running C3/C4.

In [ ]:
# ── 10a. Restore C1, C2, DSPy from Drive if missing ─────────────
for zip_name, target_dir in RESTORE_MAP.items():
    src_path = f"{DRIVE_SRC}/{zip_name}"
    target = Path(target_dir)
    if target.exists() and len(list(target.rglob("*"))) > 0:
        print(f"  ✓ {zip_name}: already on disk")
        continue
    if not Path(src_path).exists():
        print(f"  ⚠ {zip_name}: not on Drive (run C1/C2/DSPy first)")
        continue
    if target.exists():
        shutil.rmtree(target)
    target.mkdir(parents=True, exist_ok=True)
    shutil.unpack_archive(src_path, target_dir)
    print(f"  ✓ {zip_name}: restored from Drive")

# Verify DSPy
dspy_count = len(glob.glob(f"{REPO_DIR}/dspy_opt/saved/*/program.json"))
print(f"\n  DSPy programs: {dspy_count}/12")

In [ ]:
# ── 10b. Preflight check ─────────────────────────────────────────
errors = []

# Ollama
try:
    r = requests.get("http://localhost:11434/api/tags", timeout=2)
    models = [m["name"] for m in r.json().get("models", [])]
    if any(MODEL_NAME.split(":")[0] in m for m in models):
        print(f"  ✓ Ollama: {MODEL_NAME}")
    else:
        errors.append(f"Model {MODEL_NAME} not found")
except Exception:
    errors.append("Ollama not responding")

# ChromaDB
try:
    client = chromadb.PersistentClient(path=CHROMA_DB)
    chunks = sum(c.count() for c in client.list_collections())
    del client
    if chunks > 0:
        print(f"  ✓ ChromaDB: {chunks} chunks")
    else:
        errors.append("ChromaDB empty")
except Exception as e:
    errors.append(f"ChromaDB error: {e}")

# DSPy
dspy_count = len(glob.glob(f"{REPO_DIR}/dspy_opt/saved/*/program.json"))
print(f"  ✓ DSPy: {dspy_count} programs")

# C1/C2
for label, path in [("C1", f"{REPO_DIR}/output"), ("C2", f"{REPO_DIR}/output_adaptive")]:
    if Path(path).exists() and len(list(Path(path).rglob("*"))) > 0:
        print(f"  ✓ {label} output: present")
    else:
        errors.append(f"{label} output missing")

# System tools
for tool in ["clang", "checkpolicy"]:
    p = subprocess.run(["which", tool], capture_output=True, text=True)
    if p.stdout.strip():
        print(f"  ✓ {tool}")
    else:
        errors.append(f"{tool} not found")

if errors:
    print(f"\n❌ PREFLIGHT FAILED — {len(errors)} error(s):")
    for e in errors:
        print(f"  • {e}")
    raise RuntimeError("Fix errors above before C3/C4")
else:
    print(f"\n✓ Preflight passed — ready for C3/C4")

## 10c. Apply scoring & output fixes
> Three bugs found during analysis — patched automatically before C3/C4 run.
> 1. `lstrip("**/")` → `removeprefix("**/")` — glob scored HAL files as 0.000
> 2. `_clean_output()` missing on backend sub-modules — markdown fences in .py
> 3. `output_root` missing on architect agent — HAL files landed in wrong directory

In [ ]:
# ── Fix 1: lstrip → removeprefix in C3 and C4 scorers ───────────
for label, fpath in [("C3", f"{REPO_DIR}/multi_main_rag_dspy.py"),
                     ("C4", f"{REPO_DIR}/multi_main_c4_feedback.py")]:
    if not Path(fpath).exists():
        continue
    src = Path(fpath).read_text()
    if 'pattern.lstrip("**/")' in src:
        src = src.replace('pattern.lstrip("**/")', 'pattern.removeprefix("**/")')
        Path(fpath).write_text(src)
        print(f"  ✓ {label}: lstrip → removeprefix")
    else:
        print(f"  ✓ {label}: already patched")

# ── Fix 2: _clean_output() on backend sub-modules ───────────────
be_path = f"{REPO_DIR}/agents/rag_dspy_backend_agent.py"
if Path(be_path).exists():
    be = Path(be_path).read_text()
    patched = False
    for old, new, tag in [
        ('model_content = getattr(result, "models_code", "") or ""\n                self._write',
         'model_content = getattr(result, "models_code", "") or ""\n                model_content = self._clean_output(model_content)\n                self._write',
         "model"),
        ('sim_content = getattr(result, "simulator_code", "") or ""\n                self._write',
         'sim_content = getattr(result, "simulator_code", "") or ""\n                sim_content = self._clean_output(sim_content)\n                self._write',
         "simulator"),
    ]:
        if old in be and f"_clean_output({tag}_content)" not in be:
            be = be.replace(old, new)
            patched = True
    if patched:
        Path(be_path).write_text(be)
        print("  ✓ Backend: added _clean_output()")
    else:
        print("  ✓ Backend: already patched")

# ── Fix 3: output_root on architect agent ────────────────────────
for label, fpath, old, new in [
    ("C3", f"{REPO_DIR}/multi_main_rag_dspy.py",
     'agent = RAGDSPyArchitectAgent(**AGENT_CFG)',
     'agent = RAGDSPyArchitectAgent(**AGENT_CFG, output_root=str(OUTPUT_DIR))'),
    ("C4", f"{REPO_DIR}/multi_main_c4_feedback.py",
     'agent = RAGDSPyArchitectAgent(**AGENT_CFG)',
     'agent = RAGDSPyArchitectAgent(**AGENT_CFG, output_root=str(OUTPUT_DIR))'),
]:
    if not Path(fpath).exists():
        continue
    src = Path(fpath).read_text()
    if old in src:
        src = src.replace(old, new)
        Path(fpath).write_text(src)
        print(f"  ✓ {label}: added output_root to architect")
    else:
        print(f"  ✓ {label}: architect already patched")

print("\n✓ All fixes applied")

## 11. Run C3 — RAG + DSPy optimised prompts
> Combines RAG retrieval from AOSP ChromaDB with DSPy-optimised prompts.

In [ ]:
!rm -rf /content/code-codegen-aosp-llm-based/output_rag_dspy
start_ollama()
!python apply_chroma_fix.py
!python multi_main_rag_dspy.py

## 12. Run C4 — Feedback loop
> Generate → validate → refine. Up to 3 retry attempts per agent.

In [ ]:
!rm -rf /content/code-codegen-aosp-llm-based/output_c4_feedback
start_ollama()
!python multi_main_c4_feedback.py

## 13. Reporting & analysis

In [ ]:
!python diagnose_outputs.py
!python rescore_all_conditions.py
!python compare_matched.py
!python analyze_final.py

In [ ]:
print("=" * 80)
print("MATCHED ANALYSIS RESULTS")
print("=" * 80)
!cat experiments/results/matched_analysis.md

## 14. Export & save to Drive

In [ ]:
from google.colab import files

# Save all outputs to Drive
artifacts = {
    "output_c1":     f"{REPO_DIR}/output",
    "output_c2":     f"{REPO_DIR}/output_adaptive",
    "output_c3":     f"{REPO_DIR}/output_rag_dspy",
    "output_c4":     f"{REPO_DIR}/output_c4_feedback",
    "dspy_programs": f"{REPO_DIR}/dspy_opt/saved",
}

for name, src_dir in artifacts.items():
    if Path(src_dir).exists():
        shutil.make_archive(f"/content/{name}", "zip", src_dir)
        # Also save to Drive
        shutil.copy(f"/content/{name}.zip", f"{DRIVE_SRC}/{name}.zip")
        count = len(list(Path(src_dir).rglob("*")))
        print(f"  ✓ {name}.zip ({count} files)")
    else:
        print(f"  ⚠ {name}: not found")

# Save experiment results to Drive
shutil.copytree(f"{REPO_DIR}/experiments/results", f"{DRIVE_SRC}/experiments_results", dirs_exist_ok=True)
print("\n✓ All results saved to Drive")

# Download zips
for name in artifacts:
    zip_path = f"/content/{name}.zip"
    if Path(zip_path).exists():
        files.download(zip_path)

## 15. Utilities
> Run manually as needed — not part of the main pipeline.

In [ ]:
# Restart Ollama if server died
# start_ollama()

In [ ]:
# Clean up repo
# shutil.rmtree(REPO_DIR, ignore_errors=True)
# print("✓ Repo cleaned up")

In [ ]:
# Free disk (~300 MB)
# shutil.rmtree(AOSP_SRC_DIR, ignore_errors=True)
# print("✓ AOSP source removed")